# Chapter 3. Robust and Stochastic Model Predictive Control
# 강인 및 확률적 모델 예측 제어

Rawlings, Mayne, Diehl (2017), **2판 1쇄 제3장, 인쇄 pp.193–268**에 맞춘 한글 설명·실습입니다.
교재의 주요 절 3.1–3.9 및 모든 하위 절 **31개**를 원본 순서로 배치했습니다.

**학습 목표**
- 입력 열과 인과적 피드백 정책의 차이를 설명한다.
- 명목 안정성과 불확실성에 대한 보장을 구분한다.
- 최소–최대 문제를 작은 이산 모델에서 정확히 푼다.
- 불변 오차 집합으로 상태·입력 제약을 축소하고 튜브 MPC를 실행한다.
- 비선형 보조 제어의 실험 결과와 해석적으로 보인 오차 경계를 구분한다.
- 개별 시점·전체 구간의 위반 확률, 설계 실패 확률, 검증 표본의 불확실성을 구분한다.

- 여러 후보에서 최소 표본 여유를 고를 때 설계 실패 확률의 선택 편향을 보정한다.

**실행:** 코랩에서 `런타임 → 모두 실행`. 공통 준비 코드 셀 2개 뒤 각 실습을 독립적으로 실행할 수 있습니다.
1·2장 실행 결과나 저장소 복제가 필요하지 않습니다. CPU만 사용하며 최초 패키지 설치와 글꼴 다운로드에는 인터넷이 필요합니다.

**범위:** 이론은 원문에 근거해 한국어로 재서술했습니다. §3.1.2의 피드백 비교와 Example 3.13은 원본 모델·계수를 재현하며, 그 외는 직접 설계한 **보충 실습**입니다.
교재의 모든 수치 사례·증명·연습문제 해답을 재현한 자료는 아닙니다. 교재에서 2017년 기준으로 논의한 방법의 구성과 가정을 설명합니다.

**표기:** 실제 상태 `x`, 명목 상태 `z`, 명목 입력 `v`, 오차 `e=x−z`, 실제 입력 `u=v+Ke`.
교재의 막대가 있는 명목 기호를 `z,v`로 바꾸어 PPT와 노트북에서 일관되게 사용합니다.
`S`는 오차 집합, `ρ`는 스칼라 오차 수축 계수, `ε`는 위반 확률 목표, `β`는 표본 설계의 실패 확률입니다.

[PPT](https://github.com/JunsPark00/rawlings-model-predictive-control-notes/blob/main/chapter03_robust_stochastic_mpc/slides/chapter03_slides.pptx)
· [PDF](https://github.com/JunsPark00/rawlings-model-predictive-control-notes/blob/main/chapter03_robust_stochastic_mpc/slides/chapter03_slides.pdf)

## 실습 환경 준비

필요한 패키지·한글 글꼴과 공통 함수를 준비합니다. 솔버 실패를 숨기거나 입력을 임의로 잘라서 성공으로 처리하지 않습니다.

In [ ]:
import importlib.util, subprocess, sys

packages = {
    "numpy": "numpy>=2,<3",
    "scipy": "scipy>=1.14,<2",
    "matplotlib": "matplotlib>=3.9,<4",
    "cvxpy": "cvxpy>=1.6,<2",
    "osqp": "osqp>=1,<2",
}
missing = [
    spec for name, spec in packages.items() if importlib.util.find_spec(name) is None
]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import numpy as np
import scipy
from scipy.linalg import solve_discrete_are, expm
from scipy.signal import cont2discrete
import matplotlib
import matplotlib.pyplot as plt
import cvxpy as cp
from pathlib import Path
import tempfile
import urllib.request
from matplotlib import font_manager

# Colab과 로컬에서 동일하게 한글 그래프를 표시하기 위한 공개 글꼴.
font_dir = Path(tempfile.gettempdir()) / "mpc_korean_fonts"
font_dir.mkdir(exist_ok=True)
font_path = font_dir / "NanumGothic-Regular.ttf"
if not font_path.exists():
    url = "https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf"
    with urllib.request.urlopen(url, timeout=60) as response:
        font_path.write_bytes(response.read())
font_manager.fontManager.addfont(str(font_path))
korean_font = font_manager.FontProperties(fname=font_path).get_name()

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update(
    {
        "figure.figsize": (9, 3.8),
        "axes.grid": True,
        "grid.alpha": 0.22,
        "font.size": 11,
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.family": ["DejaVu Sans", korean_font],
        "mathtext.fontset": "dejavusans",
        "axes.unicode_minus": False,
    }
)
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
CHECKS = {}


def finish_figure(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=170, bbox_inches="tight")
    plt.show()
    plt.close()


def riccati(A, B, Q, R, Pf, N):
    # 부호 관례: u = K x. K에 음의 부호를 포함한다.
    Ps = [None] * (N + 1)
    Ks = [None] * N
    Ps[N] = Pf.copy()
    for k in range(N - 1, -1, -1):
        Pnext = Ps[k + 1]
        Ks[k] = -np.linalg.solve(R + B.T @ Pnext @ B, B.T @ Pnext @ A)
        Ps[k] = Q + A.T @ Pnext @ A + A.T @ Pnext @ B @ Ks[k]
        Ps[k] = (Ps[k] + Ps[k].T) / 2
    return Ps, Ks


def simulate_linear(A, B, policy, x0, steps):
    xs = [np.array(x0, dtype=float)]
    us = []
    for k in range(steps):
        u = np.atleast_1d(policy(xs[-1], k))
        us.append(u)
        xs.append(A @ xs[-1] + B @ u)
    return np.array(xs), np.array(us)


def checked_solve(problem):
    problem.solve(solver=cp.OSQP, eps_abs=1e-8, eps_rel=1e-8, max_iter=20000)
    if problem.status != cp.OPTIMAL or not np.isfinite(problem.value):
        raise RuntimeError(f"최적화 실패: {problem.status}")
    residual = max(
        (float(np.max(c.violation())) for c in problem.constraints), default=0.0
    )
    if not np.isfinite(residual) or residual > 2e-6:
        raise RuntimeError(f"제약 잔차 초과: {residual:.3g}")
    return float(problem.value)


In [ ]:
from itertools import product
from scipy.optimize import minimize
from scipy.stats import norm, beta

CHECKS = {}


def tube_parameters():
    a, b, K, wb, xlim, ulim = 1.1, 1.0, -0.6, 0.1, 3.0, 1.0
    rho = abs(a + b * K)
    s = wb / (1 - rho)
    P = float(solve_discrete_are([[a]], [[b]], [[1.0]], [[0.2]]).item())
    Kf = -b * P * a / (0.2 + b * b * P)
    return a, b, K, wb, rho, s, xlim, ulim, P, Kf


def nominal_qp(
    initial, N=6, xlim=None, ulim=None, free_center=False, terminal_zero=False
):
    if isinstance(N, bool) or not isinstance(N, (int, np.integer)) or N < 1:
        raise ValueError("N은 양의 정수여야 합니다.")
    if not np.isfinite(initial):
        raise ValueError("초기상태는 유한해야 합니다.")
    a, b, K, wb, rho, s, X, U, P, Kf = tube_parameters()
    if xlim is None:
        xlim = X - s
    if ulim is None:
        ulim = U - abs(K) * s
    if not np.isfinite(xlim) or not np.isfinite(ulim) or min(xlim, ulim) <= 0:
        raise ValueError("명목 상태·입력 한계는 유한한 양수여야 합니다.")
    z = cp.Variable(N + 1)
    v = cp.Variable(N)
    cons = [z[1:] == a * z[:-1] + b * v, cp.abs(z) <= xlim, cp.abs(v) <= ulim]
    cons += [cp.abs(z[0] - initial) <= s] if free_center else [z[0] == initial]
    cons += [z[-1] == 0] if terminal_zero else [cp.abs(z[-1]) <= 0.4]
    p = cp.Problem(
        cp.Minimize(
            cp.sum_squares(z[:-1]) + 0.2 * cp.sum_squares(v) + P * cp.square(z[-1])
        ),
        cons,
    )
    checked_solve(p)
    zz = z.value.copy()
    vv = v.value.copy()
    assert np.max(np.abs(zz)) <= xlim + 2e-6 and np.max(np.abs(vv)) <= ulim + 2e-6
    assert np.max(np.abs(zz[1:] - a * zz[:-1] - b * vv)) < 2e-6
    assert abs(zz[-1]) < (2e-6 if terminal_zero else 0.4 + 2e-6)
    assert abs(zz[0] - initial) < (s + 2e-6 if free_center else 2e-6)
    return zz, vv, float(p.value)


def error_paths(rng, M, T, rho=0.5, wb=0.1, gaussian=False):
    if any(
        isinstance(v, bool) or not isinstance(v, (int, np.integer)) or v < 1
        for v in (M, T)
    ):
        raise ValueError("경로 수와 길이는 양의 정수여야 합니다.")
    if not np.isfinite(rho) or not np.isfinite(wb) or wb < 0:
        raise ValueError("오차 계수는 유한하고 외란 크기는 비음수여야 합니다.")
    w = rng.normal(0, wb, (M, T)) if gaussian else rng.uniform(-wb, wb, (M, T))
    e = np.zeros((M, T + 1))
    for k in range(T):
        e[:, k + 1] = rho * e[:, k] + w[:, k]
    return e, w


def scenario_margin(seed=31, T=12, eps=0.05, design_beta=0.01):
    if not 0 < eps < 1 or not 0 < design_beta < 1:
        raise ValueError("epsilon과 beta는 0과 1 사이여야 합니다.")
    # One scalar decision: the largest absolute error over the whole horizon.
    M = int(np.ceil(2 / eps * (np.log(1 / design_beta) + 1)))
    e, _ = error_paths(np.random.default_rng(seed), M, T)
    margin = float(np.max(np.abs(e)))
    # Exact distribution-free order-statistic bound for this max-of-iid-paths problem.
    assert (1 - eps) ** M <= design_beta
    return margin, M


print("제3장 공통 함수 준비 완료. 각 실습은 다른 실습의 결과에 의존하지 않습니다.")


def binomial_upper(k, n, confidence=0.95):
    """고정 설계를 독립 검증한 이항 비율의 단측 Clopper–Pearson 상한."""
    if (
        any(isinstance(v, bool) or not isinstance(v, (int, np.integer)) for v in (k, n))
        or n < 1
        or not 0 <= k <= n
    ):
        raise ValueError("0 <= k <= n, n >= 1인 정수가 필요합니다.")
    if not 0 < confidence < 1:
        raise ValueError("신뢰 수준은 0과 1 사이여야 합니다.")
    return 1.0 if k == n else float(beta.ppf(confidence, k + 1, n - k))


## 3.1 Introduction — 개요

**교재 인쇄 pp.193–204**

불확실성이 있으면 하나의 예측 궤적만으로 실제 미래를 나타낼 수 없습니다. 명목 문제를 반복해서 푸는 피드백과, 예측 구간 안에서 미래 관측에 따라 행동을 바꾸도록 설계하는 피드백 정책을 구분합니다.

### 3.1.1 Types of Uncertainty — 불확실성의 종류

**교재 인쇄 pp.193–195**

외란, 파라미터 오차, 모델 구조 오차, 측정 오차는 서로 다른 경로로 제어에 영향을 줍니다. 가산 외란으로 단순화할 때에는 그 범위가 모델 오차를 실제로 포함하는지 확인해야 합니다. 유계 외란이라는 가정과 가우스 분포 가정은 같지 않습니다.

### 3.1.2 Feedback Versus Open-Loop Control — 피드백과 개방루프 제어

**교재 인쇄 pp.195–200**

개방루프 입력 열은 앞으로 발생할 외란을 관측해도 구간 내 입력을 바꾸지 않습니다. 정책은 이미 관측한 상태나 외란 이력에 따라 다음 입력을 바꿀 수 있습니다. 같은 과거를 가진 시나리오는 같은 입력을 써야 하며 아직 발생하지 않은 외란을 보고 입력을 선택하면 안 됩니다.

**원본 재현 — §3.1.2, pp.195–199, Figure 3.1**

원본의 적분기 $x^+=x+u+w$, 초기상태 1, 예측 길이 3과 비용의 공통 계수 $1/2$를 유지합니다. 외란 없는 최적 입력 열은 $(-8,-3,-1)/13$이고, 남은 구간에 따른 피드백 이득은 $(-8/13,-3/5,-1/2)$입니다. 무외란에서는 두 방식이 같지만, 외란이 생기면 피드백은 **현재 관측한 상태**에 반응합니다. 아래 이득은 명목 비용으로 설계했으며 최소–최대 최적 정책이라고 부르지 않습니다.

원본의 세 외란 경로 $(0,0,0),(1,1,1),(-1,-1,-1)$를 재현하고 8개 꼭짓점 경로로 튜브의 폭을 검증합니다. 마지막 폭은 개방루프 6, 피드백 3.4입니다. 원본 p.195 전개식의 상수항 $3x^2/2$는 나열된 네 상태 제곱을 전개하면 $2x^2$가 됩니다. 입력 최적해는 이 상수항에 영향받지 않으며 여기서는 원래 비용합을 직접 계산합니다.

In [ ]:
def demo_book_feedback():
    gains = np.array([-8 / 13, -3 / 5, -1 / 2])
    fixed = np.array([-8, -3, -1]) / 13

    def run(w, feedback):
        x = [1.0]
        u = []
        for k, wk in enumerate(w):
            uk = gains[k] * x[-1] if feedback else fixed[k]
            u.append(uk)
            x.append(x[-1] + uk + wk)
        return np.array(x), np.array(u)

    expected = [
        np.array([1, 5 / 13, 2 / 13, 1 / 13]),
        np.array([1, 18 / 13, 101 / 65, 231 / 130]),
        np.array([1, -8 / 13, -81 / 65, -211 / 130]),
    ]
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    for j, feedback in enumerate([False, True]):
        paths = np.array([run(w, feedback)[0] for w in product([-1, 1], repeat=3)])
        ax[j].fill_between(
            range(4), paths.min(0), paths.max(0), alpha=0.15, label="꼭짓점 외란의 범위"
        )
        for i, w in enumerate([np.zeros(3), np.ones(3), -np.ones(3)]):
            x, u = run(w, feedback)
            if feedback:
                assert np.allclose(x, expected[i])
            ax[j].plot(x, "o-", label=f"w = {int(w[0])}")
        ax[j].set(
            title=["고정 입력 열", "현재 상태 피드백"][j],
            xlabel="시간 단계",
            ylabel="상태",
        )
        ax[j].set_ylim(-3.2, 3.2)  # 동일 축 범위로 튜브 폭을 직접 비교
        ax[j].legend(fontsize=9)
        width = np.ptp(paths[:, -1])
        assert abs(width - [6, 3.4][j]) < 1e-12
        CHECKS[["원본 개방루프 종단 폭", "원본 피드백 종단 폭"][j]] = width
    x, u = run(np.zeros(3), True)
    cost = 0.5 * (np.sum(x * x) + np.sum(u * u))
    assert abs(cost - 21 / 26) < 1e-12
    assert np.allclose(u, fixed)
    CHECKS["원본 무외란 비용"] = cost
    finish_figure("book_feedback")
    return gains, fixed


demo_book_feedback()


### 3.1.3 Robust and Stochastic MPC — 강인 MPC와 확률적 MPC

**교재 인쇄 pp.200–202**

강인 MPC는 명시한 허용 외란의 모든 실현에 대해 제약을 다룹니다. 확률적 MPC는 외란 분포를 사용하여 기대 비용이나 확률 제약을 다룹니다. 확률 제약의 대상이 한 시점인지 구간 전체인지 명시해야 합니다. 허용 범위 또는 분포가 실제와 다르면 해당 보장을 그대로 적용할 수 없습니다.

$$
\forall w\in W^N:\ x_j\in X,\qquad \mathbb P(x_j\in X)\ge1-\epsilon
$$

### 3.1.4 Tubes — 튜브

**교재 인쇄 pp.202–203**

같은 초기상태에서 여러 외란을 적용하면 궤적 묶음이 생깁니다. 튜브의 각 단면은 해당 시점에 가능한 상태들을 포함합니다. 선형 가산 외란에서는 명목 중심과 오차 집합의 합으로 외접 튜브를 구성할 수 있습니다.

$$
X_j=\{z_j\}\oplus S_j,\qquad e_j=x_j-z_j
$$

In [ ]:
def demo_width():
    a, b, K, wb, rho, s, *_ = tube_parameters()
    T = 12
    ro = [0.0]
    rf = [0.0]
    for _ in range(T):
        ro.append(a * ro[-1] + wb)
        rf.append(rho * rf[-1] + wb)
    for N in range(1, 9):
        W = np.array(list(product([-wb, wb], repeat=N)))
        for c, radii in [(a, ro), (rho, rf)]:
            e = np.zeros(len(W))
            for j in range(N):
                e = c * e + W[:, j]
            assert abs(np.max(np.abs(e)) - radii[N]) < 1e-12
    assert max(rf) < s and abs(rho * s + wb - s) < 1e-12
    CHECKS["튜브 반경 꼭짓점 대조 구간 수"] = 8
    CHECKS["불변 오차 반경"] = s
    plt.figure()
    plt.plot(ro, "o-", label="개방루프")
    plt.plot(rf, "o-", label="피드백")
    plt.axhline(s, ls=":", color="gray", label="피드백 불변 반경")
    plt.xlabel("예측 단계")
    plt.ylabel("오차 구간 반경")
    plt.legend()
    finish_figure("tube_width")


demo_width()


### 3.1.5 Difference Inclusion Description of Uncertain Systems — 불확실한 시스템의 차분 포함 표현

**교재 인쇄 pp.203–204**

외란 변수를 명시하는 대신 가능한 다음 상태를 집합값 함수로 표현할 수 있습니다. 이 표현에서 강인 제약은 집합 전체가 허용 영역 안에 포함되어야 한다는 조건이 됩니다.

$$
F(x,u)=\{f(x,u,w):w\in W\},\qquad x^+\in F(x,u)
$$

## 3.2 Nominal (Inherent) Robustness — 명목 제어의 내재적 강인성

**교재 인쇄 pp.204–214**

명목 MPC가 안정적이라는 사실만으로 작은 외란에도 같은 결과를 얻는 것은 아닙니다. 값 함수와 폐루프의 연속성, 제약 경계로부터의 여유, 외란 크기 등의 가정을 함께 봐야 합니다.

### 3.2.1 Introduction — 내재적 강인성의 출발점

**교재 인쇄 pp.204–206**

교재는 불연속 폐루프에서 명목 점근 안정성이 임의로 작은 섭동에도 유지되지 않는 사례를 설명합니다. 이는 특정 시뮬레이션이 원점으로 수렴했다는 사실만으로 강인성을 주장할 수 없음을 보여 줍니다.

### 3.2.2 Difference Inclusion Description of Discontinuous Systems — 불연속 시스템의 차분 포함 표현

**교재 인쇄 pp.206–207**

불연속점 근방에서 나타나는 함수값의 극한들을 포함하여 정규화하면 작은 상태 오차로 선택될 수 있는 변화도 분석할 수 있습니다. 이 절의 정규화는 집합의 폐포를 사용하며 임의로 볼록껍질을 추가하는 정의와 구분합니다.

$$
F(x)=\bigcap_{\delta>0}\overline{\{f(y):\|y-x\|\le\delta\}}
$$

### 3.2.3 When Is Nominal MPC Robust? — 명목 MPC는 언제 강인한가?

**교재 인쇄 pp.207–209**

외란이 지속되면 정확한 원점 수렴보다 초기 편차의 감소와 외란에 따른 잔여 오차를 구분하는 것이 자연스럽습니다. 교재의 강인성 정의는 원하는 오차 수준과 초기상태의 콤팩트 집합에 맞는 충분히 작은 섭동 크기를 요구합니다.

$$
\|x_k\|\le b(\|x_0\|,k)+\epsilon,\qquad b\in\mathcal{KL}
$$

**보충 실습 — 외란이 지속되면 원점 대신 작은 집합으로 접근합니다**

$e^+=\rho e+w$, $|w|\le\bar w$, $|\rho|<1$이면 반복 대입과 삼각부등식으로
$$|e_k|\le |\rho|^k|e_0|+\bar w\frac{1-|\rho|^k}{1-|\rho|}.$$
따라서 잔여 반경은 $s=\bar w/(1-|\rho|)$입니다. 양의 $\rho,e_0$와 항상 양의 최대 외란에서는 상한이 실제로 달성됩니다. 이는 이 스칼라 선형계의 직접 증명이며 일반적인 명목 MPC의 강인성 정리 전체를 재현하지 않습니다.

In [ ]:
def demo_practical_bound():
    rho = 0.6
    e0 = 1.0
    T = 30
    t = np.arange(T + 1)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    for wb in [0, 0.04, 0.1]:
        radius = wb / (1 - rho)
        bound = rho**t * e0 + radius * (1 - rho**t)
        e = [e0]
        for _ in range(T):
            e.append(rho * e[-1] + wb)
        assert np.allclose(e, bound)
        ax[0].plot(t, e, label=f"외란 상한 {wb:g}")
        ax[0].axhline(radius, ls=":", alpha=0.5)
        ax[1].plot(t, np.maximum(np.abs(e) - radius, 0), label=f"외란 상한 {wb:g}")
    ax[0].set(xlabel="시간 단계", ylabel="최악 방향 상태")
    ax[1].set(xlabel="시간 단계", ylabel="불변 구간까지 거리", yscale="log")
    for a in ax:
        a.legend()
    CHECKS["외란 0.1의 잔여 반경"] = 0.1 / (1 - rho)
    finish_figure("practical_bound")


demo_practical_bound()


### 3.2.4 Robustness of Nominal MPC — 명목 MPC의 강인성

**교재 인쇄 pp.209–214**

값 함수 감소가 충분히 크고 섭동에 따른 변화가 작으면 원점 주변의 작은 값 함수 부분집합으로 진입하고 머무르는 결과를 얻습니다. 이론은 절에서 명시한 가정을 필요로 합니다. 아래 보충 실습은 안정성 정리의 재현이 아니라, 명목 제약 만족이 외란 후 제약 만족을 뜻하지 않는 한 단계 예입니다.

In [ ]:
def demo_nominal_boundary():
    predicted = []
    actual = []
    for margin in [0.0, 0.08]:
        u = cp.Variable()
        xp = 1.2 * 0.9 + u
        p = cp.Problem(
            cp.Minimize(100 * cp.square(u) + cp.square(xp)),
            [cp.abs(u) <= 0.3, xp <= 1 - margin, xp >= -1 + margin],
        )
        checked_solve(p)
        predicted.append(float(xp.value))
        actual.append(float(xp.value + 0.08))
    assert actual[0] > 1.07 and actual[1] <= 1 + 1e-7
    CHECKS["명목 한 단계의 실제 상한 위반량"] = actual[0] - 1
    x = np.arange(2)
    plt.figure()
    plt.bar(x - 0.17, predicted, 0.34, label="명목 예측")
    plt.bar(x + 0.17, actual, 0.34, label="외란 적용 후")
    plt.axhline(1, color="gray", ls=":", label="상태 상한")
    plt.xticks(x, ["여유 없음", "여유 0.08"])
    plt.ylabel("다음 상태")
    plt.legend()
    finish_figure("nominal_constraint")


demo_nominal_boundary()


## 3.3 Min-Max Optimal Control: Dynamic Programming Solution — 최소–최대 최적 제어와 동적 계획법

**교재 인쇄 pp.214–220**

최소–최대 문제는 제어자가 선택한 정책에 대해 가장 불리한 허용 외란을 평가합니다. 동적 계획법은 다음 상태에서 다시 제어를 선택하는 구조를 재귀에 포함합니다. 작은 이산 문제에서는 근사 격자 없이 정확히 계산할 수 있지만 일반적인 연속 고차원 문제에서는 계산량이 크게 증가합니다.

### 3.3.1 Introduction — 최소–최대 문제의 구성

**교재 인쇄 pp.214–216**

허용 정책은 모든 외란 실현에서 상태·입력·종단 제약을 만족해야 합니다. 제어 입력을 선택한 뒤 외란이 실현되는 정보 순서를 유지합니다. 외란을 먼저 알고 입력을 선택하는 계산은 더 쉬운 다른 문제입니다.

$$
V_N(x)=\min_{\pi\in\Pi_N(x)}\max_{w\in W^N}J_N(x,\pi,w)
$$

**보충 실습 — 미래 외란을 미리 보면 더 쉬운 문제를 풉니다**

한 단계 모델 $x^+=u+w$, $x_0=0$, $u,w\in\{-1,1\}$와 비용 $(x^+)^2$를 비교합니다. 실제 정보 순서인 $\min_u\max_w$의 값은 4입니다. 외란을 미리 보고 $u=-w$를 고르는 $\max_w\min_u$의 값은 0입니다. 시나리오별 입력을 모두 독립 변수로 만들면 이런 **미래 정보 사용**이 숨어들 수 있습니다. 같은 과거를 가진 시나리오의 입력에는 비예견성 제약을 넣어야 합니다.

In [ ]:
def demo_information_order():
    U = np.array([-1, 1])
    W = np.array([-1, 1])
    cost = (U[:, None] + W[None, :]) ** 2
    causal = float(cost.max(axis=1).min())
    oracle = float(cost.min(axis=0).max())
    assert causal == 4 and oracle == 0
    CHECKS["정보 순서에 따른 비용 격차"] = causal - oracle
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
    ax[0].imshow(cost, cmap="Blues", vmin=0, vmax=4)
    for i in range(2):
        for j in range(2):
            ax[0].text(
                j,
                i,
                str(cost[i, j]),
                ha="center",
                va="center",
                color="white" if cost[i, j] else "black",
            )
    ax[0].set(
        xticks=[0, 1],
        xticklabels=W,
        yticks=[0, 1],
        yticklabels=U,
        xlabel="이후 실현되는 외란 w",
        ylabel="먼저 선택하는 입력 u",
    )
    ax[1].bar(["인과적 제어", "외란을 미리 아는 제어"], [causal, oracle])
    ax[1].set(ylabel="최악 비용")
    finish_figure("information_order")


demo_information_order()


### 3.3.2 Properties of the Dynamic Programming Solution — 동적 계획법 해의 성질

**교재 인쇄 pp.216–220**

벨만 재귀는 한 단계 비용과 남은 최적 비용을 결합합니다. 아래에서는 정수 상태와 유한 입력·외란 집합을 가진 보충 문제를 정확히 풀어, 개방루프 입력 열의 최악 비용과 비교합니다. 연속 상태 공간을 격자로 근사한 결과가 아닙니다. 종단 비용은 상태 제곱이고 종단 제약은 전체 허용 상태 집합이며, 이 유한 구간 비교의 안정성을 별도로 주장하지 않습니다.

$$
V_j(x)=\min_{u}\max_{w}\left[x^2+0.2u^2+V_{j-1}(x+u+w)\right]
$$

In [ ]:
def demo_minmax():
    states = range(-4, 5)
    U = [-1, 0, 1]
    W = [-1, 1]
    N = 3
    V = [{x: float(x * x) for x in states}]
    pol = [{}]
    for h in range(1, N + 1):
        vh = {}
        ph = {}
        for x in states:
            options = []
            for u in U:
                val = max(x * x + 0.2 * u * u + V[-1].get(x + u + w, np.inf) for w in W)
                options.append((val, u))
            vh[x], ph[x] = min(options)
        V.append(vh)
        pol.append(ph)
    closed = []
    opened = []
    for x0 in range(-2, 3):
        scores = []
        for ws in product(W, repeat=N):
            x = x0
            j = 0.0
            for k, w in enumerate(ws):
                u = pol[N - k][x]
                j += x * x + 0.2 * u * u
                x = x + u + w
                assert x in states
            scores.append(j + x * x)
        assert abs(max(scores) - V[N][x0]) < 1e-10
        candidates = []
        for us in product(U, repeat=N):
            worst = 0.0
            for ws in product(W, repeat=N):
                x = x0
                j = 0.0
                for u, w in zip(us, ws):
                    j += x * x + 0.2 * u * u
                    x = x + u + w
                    if x not in states:
                        j = np.inf
                        break
                worst = max(worst, j + x * x)
            candidates.append(worst)
        vopen = min(candidates)
        assert V[N][x0] <= vopen + 1e-10
        closed.append(V[N][x0])
        opened.append(vopen)
    assert max(np.array(opened) - closed) > 0.1
    CHECKS["정책 사용의 최악 비용 개선 최댓값"] = float(max(np.array(opened) - closed))
    plt.figure()
    plt.plot(range(-2, 3), closed, "o-", label="인과적 피드백 정책")
    plt.plot(range(-2, 3), opened, "s--", label="고정 입력 열")
    plt.xlabel("초기 정수 상태")
    plt.ylabel("최악 구간 비용")
    plt.legend()
    finish_figure("minmax_dp")


demo_minmax()


## 3.4 Robust Min-Max MPC — 강인 최소–최대 MPC

**교재 인쇄 pp.220–223**

일반적인 함수 정책 전체를 최적화하기는 어렵기 때문에 고정 피드백 이득과 조정 가능한 항 등으로 정책을 매개변수화합니다. 선택한 정책 집합 안에서의 최적해는 모든 인과적 정책 중의 최적해와 다를 수 있습니다. 지속 외란 아래 안정화 논리는 불변 목표 집합과 적절한 비용·종단 조건을 필요로 합니다.

## 3.5 Tube-Based Robust MPC — 튜브 기반 강인 MPC

**교재 인쇄 pp.223–236**

튜브 MPC는 명목 계획과 오차를 억제하는 보조 피드백을 결합합니다. 불변 오차 집합을 먼저 계산하여 명목 문제의 제약을 축소하면 온라인에서는 보통의 명목 최적화 구조를 활용할 수 있습니다. 이는 최소–최대 정책 문제 전체의 최적해를 그대로 계산하는 방법은 아닙니다.

$$
z^+=Az+Bv,\qquad u=v+K(x-z),\qquad e^+=(A+BK)e+w
$$

### 3.5.1 Introduction — 튜브 MPC의 구성

**교재 인쇄 pp.223–224**

명목 중심은 외란 없는 모델로 이동하고 실제 상태와의 차이는 보조 피드백으로 처리합니다. 실제 상태가 변했다고 매번 명목 중심을 무조건 실제 상태로 재설정하면 기존 실행 가능성 논리를 잃을 수 있습니다.

### 3.5.2 Outer-Bounding Tube for a Linear System with Additive Disturbance — 선형 가산 외란 시스템의 외접 튜브

**교재 인쇄 pp.224–228**

오차 집합 S가 (A+BK)S와 W의 민코프스키 합을 포함하면 강인 양의 불변 집합입니다. 실제 상태와 입력이 원래 한계를 지키도록 명목 상태와 입력에 여유를 둡니다. 스칼라 대칭 구간에서는 불변 반경과 제약 축소를 해석적으로 계산할 수 있습니다.

$$
S=[-s,s],\quad s=\frac{w_{\max}}{1-|a+bK|},\quad |z|\le x_{\max}-s,\quad |v|\le u_{\max}-|K|s
$$

**보충 실습 — 오차 반경과 입력 여유의 동시 설계**

이득 $K$를 바꾸면 $s=\bar w/(1-|a+bK|)$와 보정 입력 여유 $|K|s$가 함께 변합니다. 작은 반경만 보고 이득을 고르면 명목 제어에 남는 입력 $U-|K|s$를 놓칩니다. 아래에서는 안정한 이득만 평가하고, 상태·입력 한계가 양수인 경우에만 축소 문제를 설계할 수 있음을 확인합니다. $K$마다 종단 조건을 별도로 확인해야 하며 이 그림 자체는 폐루프 성능 최적화가 아닙니다.

In [ ]:
def demo_gain_tradeoff():
    a, b, _, wb, _, _, X, U, *_ = tube_parameters()
    gains = np.linspace(-1.8, -0.2, 161)
    rho = np.abs(a + b * gains)
    assert np.all(rho < 1)
    s = wb / (1 - rho)
    reserve = np.abs(gains) * s
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    ax[0].plot(gains, s, label="오차 반경 s")
    ax[0].plot(gains, reserve, label="보정 입력 여유 |K|s")
    ax[1].plot(gains, X - s, label="명목 상태 한계")
    ax[1].plot(gains, U - reserve, label="명목 입력 한계")
    for a in ax:
        a.set(xlabel="보조 이득 K")
        a.legend()
    CHECKS["이득 설계 최소 명목 입력 한계"] = float((U - reserve).min())
    assert np.all(U - reserve > 0)
    finish_figure("gain_tradeoff")


demo_gain_tradeoff()


### 3.5.3 Tube-Based MPC of Linear Systems with Additive Disturbances — 선형 가산 외란 시스템의 튜브 MPC

**교재 인쇄 pp.228–234**

명목 QP에는 축소된 상태·입력 제약과 안정화 종단 조건을 사용합니다. 명목 중심을 다음 상태로 이동하면 기존 최적 입력 열을 이동한 후보가 다음 문제의 실행 가능성을 보장합니다. 아래 실습에서는 보조 피드백 K=-0.6과 명목 종단 LQR 이득 Kf를 구분합니다. 실제 상태는 원점 자체가 아니라 원점 주변 오차 집합에 접근합니다.

**원본 재현 — Example 3.13, pp.231–233: 2차원 제약 축소**

원본의 $A=\begin{bmatrix}1&1\\0&1\end{bmatrix}$, $B=(0,1)^T$, $K=(-0.4,-1.2)$, $W=[-0.1,0.1]^2$, $|x_i|\le1$, $|u|\le1$을 사용합니다. $A_K=A+BK$와 오차 집합 $S_N=\bigoplus_{j=0}^{N-1}A_K^jW$를 정의합니다. 여기서 $N$은 **오프라인 집합 근사 길이**이며 온라인 MPC 예측 길이와 별개입니다.

상자 외란의 지지 함수는 $h_W(c)=0.1\|c\|_1$입니다. 따라서 상태·입력 각 행 $c\in\{(1,0),(0,1),K\}$의 누적 최악 오차는 $\theta_N(c)=\sum_{j=0}^{N-1}h_W(cA_K^j)$입니다. 원본처럼 $A_K^NW\subseteq\alpha W$와 $KA_K^NW\subseteq\alpha KW$를 함께 확인합니다. $\alpha<1$이면 $\theta_\infty\le\theta_N/(1-\alpha)$로 외접 여유를 얻습니다.

**핵심:** 유한 합 $S_N$은 일반적으로 무한 불변 집합의 내부 근사입니다. 외접 보정 없이 강인 보장용으로 쓰면 안 됩니다. 원본처럼 처음 $\alpha<1$인 길이는 3이지만, 모든 축소 한계가 비음수인 길이는 4입니다. 아래는 Figure 3.4–3.5의 계산 구조를 재현하며 그림의 픽셀을 복사하지 않습니다.

In [ ]:
def demo_book_tightening():
    A = np.array([[1.0, 1.0], [0.0, 1.0]])
    B = np.array([[0.0], [1.0]])
    K = np.array([[-0.4, -1.2]])
    AK = A + B @ K
    rows = np.vstack([np.eye(2), K])
    wb = 0.1
    vertices = wb * np.array(list(product([-1.0, 1.0], repeat=2)))
    alpha = []
    bounds = []
    theta = np.zeros(3)
    power = np.eye(2)
    for N in range(1, 26):
        theta += wb * np.abs(rows @ power).sum(axis=1)
        power = power @ AK
        aa = max(
            np.abs(power).sum(axis=1).max(), np.abs(K @ power).sum() / np.abs(K).sum()
        )
        by_vertices = max(
            np.abs(vertices @ power.T).max() / wb,
            np.abs(vertices @ (K @ power).T).max() / (wb * np.abs(K).sum()),
        )
        assert abs(aa - by_vertices) < 1e-12
        alpha.append(aa)
        bounds.append(1 - theta / (1 - aa) if aa < 1 else [np.nan] * 3)
    alpha = np.array(alpha)
    bounds = np.array(bounds)
    first = int(np.flatnonzero(alpha < 1)[0] + 1)
    nonempty = int(np.flatnonzero(np.all(bounds >= 0, axis=1))[0] + 1)
    assert first == 3 and nonempty == 4
    # 200 항의 내부 합에 기하급수 꼬리 상한을 더해 외접 근사를 독립 대조.
    finite = np.zeros(3)
    power = np.eye(2)
    for _ in range(200):
        finite += wb * np.abs(rows @ power).sum(axis=1)
        power = power @ AK
    # A_K^3이 무한대 노름에서 수축하므로 3항 묶음으로 무한 꼬리를 제한.
    block_norm = np.linalg.norm(np.linalg.matrix_power(AK, 3), ord=np.inf)
    assert block_norm < 1
    block_sum = sum(
        np.linalg.norm(np.linalg.matrix_power(AK, j), ord=np.inf) for j in range(3)
    )
    tail = wb * np.abs(rows @ power).sum(axis=1) * block_sum / (1 - block_norm)
    assert np.max(tail) < 1e-40
    assert np.all(1 - bounds[alpha < 1] >= finite + tail - 1e-12)
    CHECKS["원본 alpha 조건 최초 길이"] = first
    CHECKS["원본 축소 집합 비공집합 최초 길이"] = nonempty
    CHECKS["원본 근사 25항 명목 위치 한계"] = float(bounds[-1, 0])
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    ns = np.arange(1, 26)
    ax[0].semilogy(ns, alpha, "o-")
    ax[0].axhline(1, color="gray", ls=":")
    ax[0].set(xlabel="오프라인 근사 길이 N", ylabel="외접 보정 계수 alpha")
    for j, name in enumerate(["상태 1", "상태 2", "입력"]):
        ax[1].plot(ns, bounds[:, j], label=name)
    ax[1].axhline(0, color="gray", ls=":")
    ax[1].set(xlabel="오프라인 근사 길이 N", ylabel="축소된 명목 한계")
    ax[1].legend()
    finish_figure("book_tightening")
    return AK, rows, bounds


demo_book_tightening()


In [ ]:
def demo_robust_tube():
    a, b, K, wb, rho, s, X, U, P, Kf = tube_parameters()
    T = 30
    assert abs(Kf) * 0.4 <= U - abs(K) * s and abs(a + b * Kf) * 0.4 <= 0.4
    assert abs(rho * s + wb - s) < 1e-12
    _, W = error_paths(np.random.default_rng(2), 400, T)
    W = np.vstack([W, np.full(T, wb), np.full(T, -wb)])
    actual = np.zeros((len(W), T + 1))
    actual[:, 0] = 2.5
    Z = [2.5]
    inputs = []
    res = []
    for k in range(T):
        z, v, value = nominal_qp(Z[-1])
        err = actual[:, k] - Z[-1]
        u = v[0] + K * err
        actual[:, k + 1] = a * actual[:, k] + b * u + W[:, k]
        Z.append(a * Z[-1] + b * v[0])
        inputs.append(u)
        shifted = np.r_[v[1:], Kf * z[-1]]
        zz = np.r_[Z[-1], np.zeros(len(shifted))]
        for j, uu in enumerate(shifted):
            zz[j + 1] = a * zz[j] + uu
        assert (
            np.max(np.abs(zz)) <= X - s + 2e-6
            and np.max(np.abs(shifted)) <= U - abs(K) * s + 2e-6
        )
        assert abs(zz[-1]) <= 0.4 + 2e-6
        candidate = np.sum(zz[:-1] ** 2) + 0.2 * np.sum(shifted**2) + P * zz[-1] ** 2
        residual = candidate - value + z[0] ** 2 + 0.2 * v[0] ** 2
        assert residual < 2e-6
        res.append(residual)
    Z = np.array(Z)
    inputs = np.array(inputs).T
    error = actual - Z
    assert (
        np.max(np.abs(error)) <= s + 2e-6
        and np.max(np.abs(actual)) <= X + 2e-6
        and np.max(np.abs(inputs)) <= U + 2e-6
    )
    assert abs(Z[-1]) < 1e-5
    CHECKS["강인 튜브 오차 경계 초과량"] = float(max(0, np.max(np.abs(error)) - s))
    CHECKS["강인 튜브 상태 최대 크기"] = float(np.max(np.abs(actual)))
    CHECKS["강인 튜브 입력 최대 크기"] = float(np.max(np.abs(inputs)))
    CHECKS["명목 이동 후보 비용 감소 최대 잔차"] = float(max(res))
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    t = np.arange(T + 1)
    ax[0].fill_between(t, Z - s, Z + s, alpha=0.2, label="보장 튜브")
    ax[0].plot(t, actual[:40].T, lw=0.5, alpha=0.25)
    ax[0].plot(t, Z, color="black", label="명목 중심")
    ax[0].set(xlabel="시간 단계", ylabel="상태")
    ax[0].legend()
    ax[1].fill_between(
        range(T), inputs.min(0), inputs.max(0), alpha=0.35, label="실제 입력 범위"
    )
    for h in [-U, U]:
        ax[1].axhline(h, color="gray", ls=":")
    ax[1].set(xlabel="시간 단계", ylabel="입력")
    ax[1].legend()
    finish_figure("robust_tube")


demo_robust_tube()


### 3.5.4 Improved Tube-Based MPC of Linear Systems with Additive Disturbances — 명목 중심을 최적화하는 개선된 튜브 MPC

**교재 인쇄 pp.234–236**

현재 실제 상태가 새 튜브 안에 있다는 조건을 유지하면서 명목 초기상태도 최적화할 수 있습니다. 기존 중심이 후보로 남으면 최적 명목 비용은 그 후보보다 커지지 않습니다. 아래 스칼라 실습은 정확한 불변 구간을 사용하는 보충 구성입니다. 명목 비용 개선과 실제 누적 비용 개선은 같은 주장이 아닙니다.

$$
\min_{z_0,v}V_N(z_0,v)\quad\mathrm{s.t.}\quad x-z_0\in S
$$

In [ ]:
def demo_center():
    a, b, K, wb, rho, s, X, U, P, Kf = tube_parameters()
    x = 2.9
    old = 2.7
    xs = [x]
    centers = []
    gaps = []
    assert x > X - s and abs(x - old) <= s + 1e-10
    for k in range(25):
        _, _, baseline = nominal_qp(old)
        z, v, value = nominal_qp(x, free_center=True)
        assert value <= baseline + 2e-6
        u = v[0] + K * (x - z[0])
        assert abs(u) <= U + 2e-6
        w = wb * np.sin(1.1 * k)
        x = a * x + b * u + w
        old = a * z[0] + b * v[0]
        assert abs(x - old) <= s + 2e-6 and abs(x) <= X + 2e-6
        centers.append(z[0])
        gaps.append(max(0, baseline - value))
        xs.append(x)
    CHECKS["최적 중심 선택의 명목 비용 개선 최댓값"] = float(max(gaps))
    assert max(gaps) > 0.01
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    ax[0].plot(xs, label="실제 상태")
    ax[0].plot(centers, "--", label="선택한 중심")
    ax[0].set(xlabel="시간 단계", ylabel="상태")
    ax[0].legend()
    ax[1].plot(gaps, "o-")
    ax[1].set(xlabel="시간 단계", ylabel="기존 중심 대비 명목 비용 개선")
    finish_figure("optimized_center")


demo_center()


**보충 실습 — 모델 오차를 외란 집합에 포함하지 않으면 튜브가 깨집니다**

실제 계수가 $a+\Delta a$이면 $e^+=(a+bK)e+w+\Delta a\,x$입니다. 원래 외란 $|w|\le0.1$만 사용한 튜브는 모델 오차를 포함하지 않습니다. $|x|\le X$라는 **조건 아래** 새 가산 외란 상한은 $0.1+|\Delta a|X$입니다. 이 상한으로 튜브를 다시 만들 때에는 상태·입력 여유와 종단 설계까지 다시 확인해야 합니다.

아래는 $x=2.5,e=0.2,\Delta a=0.04$인 한 단계 경계 반례입니다. 재설계 반경은 조건부 오차 상한 계산이며 완성된 강인 MPC의 증명이라고 해석하지 않습니다.

In [ ]:
def demo_model_mismatch():
    a, b, K, wb, rho, s, X, U, *_ = tube_parameters()
    delta = 0.04
    x = 2.5
    e = s
    wrong = rho * e + wb + delta * x
    effective = wb + abs(delta) * X
    new_s = effective / (1 - rho)
    assert wrong > s and rho * new_s + effective <= new_s + 1e-12
    CHECKS["모델 오차 누락의 한 단계 튜브 위반"] = wrong - s
    CHECKS["모델 오차 포함 조건부 반경"] = new_s
    plt.figure()
    plt.bar(
        ["기존 설계 반경", "모델 오차 후 실제 오차", "재설계 조건부 반경"],
        [s, wrong, new_s],
    )
    plt.ylabel("오차 크기")
    finish_figure("model_mismatch")


demo_model_mismatch()


## 3.6 Tube-Based MPC of Nonlinear Systems — 비선형 시스템의 튜브 MPC

**교재 인쇄 pp.236–246**

비선형 모델의 오차 변화는 일반적으로 오차 하나만의 선형식으로 분리되지 않습니다. 교재는 명목 경로를 따라가는 보조 MPC와 시변 값 함수의 부분집합으로 튜브를 설명합니다. 선형 불변 집합 계산을 그대로 대입할 수는 없습니다.

$$
e^+=f(x,u,w)-f(z,v,0)
$$

### 3.6.1 The Nominal Trajectory — 명목 궤적

**교재 인쇄 pp.238**

명목 경로는 원래 제약에서 여유를 둔 상태·입력 한계를 지키도록 생성합니다. 이 절의 구성에서는 명목 종단 상태를 0으로 고정하고 이후 명목 상태와 입력을 0으로 연장합니다. 명목 경로가 실행 가능하다는 사실과 실제 불확실한 경로의 제약 만족은 구분해야 합니다.

### 3.6.2 Model Predictive Controller — 명목 경로를 추종하는 보조 MPC

**교재 인쇄 pp.238–242**

보조 MPC는 현재 실제 상태에서 시작하여 명목 경로와의 상태·입력 편차 비용을 최소화합니다. 교재의 이 구성은 입력 제약을 직접 사용하며, 상태·종단 조건은 가중 종단 비용과 부분집합 논리로 분석합니다. 아래의 유한 구간 SLSQP 실습은 이 추종 구조를 보여 주는 보충 예이며 교재 정리의 모든 상수를 검증한 것은 아닙니다.

$$
J_a=\sum_{j=0}^{H-1}\left(\|x_j-z_{t+j}\|_Q^2+\|u_j-v_{t+j}\|_R^2\right)+V_f(x_H)
$$

### 3.6.3 Choosing the Nominal Constraint Sets Ū and X̄ — 명목 입력·상태 제약 집합의 선택

**교재 인쇄 pp.242–246**

명목 제어에 사용할 입력과 외란 억제를 위한 입력 여유를 나누고, 실제 튜브가 상태 제약 안에 놓이도록 명목 한계를 선택합니다. 아래에서는 보조 MPC를 수치적으로 검사하고, 별도의 비선형 상쇄 피드백에 대해서는 스칼라 오차 경계를 해석적으로 보입니다. 상쇄 피드백의 증명을 보조 MPC의 보장으로 옮겨 쓰지 않습니다.

**보충 실습 — 보조 MPC와 해석적 상쇄 피드백의 구분**

명목 경로는 $|z|\le1.2$, $|v|\le0.7$, $z_8=0$으로 구합니다. 보조 MPC는 예측 길이 5, 입력 한계 1을 사용하며 종단에도 명목 경로와의 편차 $10(x_H-z_{t+H})^2$를 벌점화합니다.
이는 교재 정리의 모든 가정을 재현한 구성이 아니라 추종 구조를 확인하는 보충 실험입니다.
별도의 상쇄 제어는
$$u=v+2\{0.4(x-z)-0.85(x-z)-0.05(x^3-z^3)\}$$
이며 $e^+=0.4e+w$를 정확히 만듭니다. $s=1/30$이면 $0.4s+0.02=s$입니다.
$|z|\le1.2, |e|\le s$에서 평균값 정리를 사용하면
$$|u-v|\le 2\{0.45+0.15(1.2+s)^2\}s$$
이므로 명목 입력 0.7에 이 여유를 더해도 실제 입력 한계 1보다 작습니다.
상쇄 제어는 모델이 정확하다는 가정을 사용하며 교재의 보조 MPC 자체는 아닙니다.

In [ ]:
def demo_nonlinear():
    def f(x, u):
        return 0.85 * x + 0.05 * x**3 + 0.5 * u

    def rollout(x, u):
        out = [x]
        for uk in u:
            out.append(f(out[-1], uk))
        return np.array(out)

    N = 8
    seed = []
    z = 0.8
    for k in range(N):
        v = -1.2 * z if k < N - 1 else -2 * (0.85 * z + 0.05 * z**3)
        seed.append(v)
        z = f(z, v)
    objective = lambda u: np.sum(rollout(0.8, u)[:-1] ** 2) + 0.2 * np.sum(u * u)
    sol = minimize(
        objective,
        np.array(seed),
        method="SLSQP",
        bounds=[(-0.7, 0.7)] * N,
        constraints=[
            {"type": "eq", "fun": lambda u: rollout(0.8, u)[-1]},
            {"type": "ineq", "fun": lambda u: 1.2 - np.abs(rollout(0.8, u))},
        ],
        options={"ftol": 1e-11, "maxiter": 300},
    )
    assert sol.success, sol.message
    z = rollout(0.8, sol.x)
    assert abs(z[-1]) < 1e-7 and np.max(np.abs(z)) <= 1.2 + 1e-7
    T = 24
    H = 5
    Z = np.r_[z, np.zeros(T + H)]
    V = np.r_[sol.x, np.zeros(T + H)]
    xm = 0.8
    xc = 0.8
    mp = [xm]
    cancel = [xc]
    max_input = 0.0
    s = 0.02 / 0.6
    reserve = 2 * (0.45 + 0.15 * (1.2 + s) ** 2) * s
    assert 0.7 + reserve < 1 and 1.2 + s < 1.5
    for t in range(T):
        targetz = Z[t : t + H + 1]
        targetv = V[t : t + H]

        def cost(u):
            xx = rollout(xm, u)
            return (
                np.sum((xx[:-1] - targetz[:-1]) ** 2)
                + 0.2 * np.sum((u - targetv) ** 2)
                + 10 * (xx[-1] - targetz[-1]) ** 2
            )

        opt = minimize(
            cost,
            targetv.copy(),
            method="SLSQP",
            bounds=[(-1, 1)] * H,
            options={"ftol": 1e-10, "maxiter": 150},
        )
        assert opt.success, opt.message
        assert opt.fun <= cost(targetv) + 1e-7
        um = opt.x[0]
        e = xc - Z[t]
        uc = V[t] + 2 * (0.4 * e - 0.85 * e - 0.05 * (xc**3 - Z[t] ** 3))
        w = 0.02 * np.sin(1.7 * t)
        xm = f(xm, um) + w
        xc = f(xc, uc) + w
        assert abs((xc - Z[t + 1]) - (0.4 * e + w)) < 1e-7
        assert abs(xc - Z[t + 1]) <= s + 1e-7 and abs(uc) <= 1 + 1e-7
        max_input = max(max_input, abs(um), abs(uc))
        mp.append(xm)
        cancel.append(xc)
    assert max(np.abs(mp)) <= 1.5 + 1e-6
    CHECKS["비선형 명목 종단 등식 잔차"] = abs(float(z[-1]))
    CHECKS["비선형 상쇄 피드백 입력 여유 상한"] = reserve
    CHECKS["보조 MPC 실험 최대 추종 오차"] = float(
        np.max(np.abs(np.array(mp) - Z[: T + 1]))
    )
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    tt = np.arange(T + 1)
    ax[0].plot(Z[: T + 1], "k--", label="명목 경로")
    ax[0].plot(mp, label="보조 MPC")
    ax[0].plot(cancel, ":", label="상쇄 피드백")
    ax[0].set(xlabel="시간 단계", ylabel="상태")
    ax[0].legend()
    ax[1].fill_between(tt, -s, s, alpha=0.2, label="상쇄 피드백 보장 범위")
    ax[1].plot(np.array(mp) - Z[: T + 1], label="보조 MPC 오차")
    ax[1].plot(np.array(cancel) - Z[: T + 1], ":", label="상쇄 피드백 오차")
    ax[1].set(xlabel="시간 단계", ylabel="추종 오차")
    ax[1].legend(fontsize=8)
    finish_figure("nonlinear_tube")


demo_nonlinear()


## 3.7 Stochastic MPC — 확률적 MPC

**교재 인쇄 pp.246–257**

확률적 MPC는 외란의 확률 모델을 사용합니다. 기대 비용을 최소화하는 것과 위반 확률을 제한하는 것은 다른 목표입니다. 무계 외란에 선형 오차 피드백을 쓰면서 모든 상태·입력에 유계의 강제약을 요구하면 양립할 수 없는 경우가 생깁니다.

### 3.7.1 Introduction — 확률적 제약의 의미

**교재 인쇄 pp.246–248**

각 시점에서 95% 제약 만족을 보장해도 구간 전체가 95% 확률로 안전하다는 뜻은 아닙니다. 결합 사건을 직접 다루거나 합집합 상계로 전체 위험을 배분해야 합니다. 다음 실습의 가우스 오차는 분산 재귀가 정확한 별도 예이며, 뒤의 유계 외란 튜브 제어와 가정을 구분합니다.

$$
\sigma_{j+1}^2=\rho^2\sigma_j^2+\sigma_w^2,\qquad m_j=\Phi^{-1}(1-\epsilon_j/2)\sigma_j,\qquad\sum_j\epsilon_j\le\epsilon
$$

In [ ]:
def demo_gaussian():
    M, T, eps = 30000, 20, 0.05
    rho = 0.5
    sig = 0.1
    E, _ = error_paths(np.random.default_rng(41), M, T, rho, sig, gaussian=True)
    var = np.zeros(T + 1)
    for j in range(T):
        var[j + 1] = rho * rho * var[j] + sig * sig
    point = norm.ppf(1 - eps / 2) * np.sqrt(var)
    joint = norm.ppf(1 - eps / (2 * T)) * np.sqrt(var)
    rate_point = (np.abs(E[:, 1:]) > point[1:]).mean(0)
    joint_bad = np.any(np.abs(E[:, 1:]) > joint[1:], axis=1)
    point_path = np.any(np.abs(E[:, 1:]) > point[1:], axis=1).mean()
    assert np.max(np.abs(rate_point - eps)) < 6 * np.sqrt(eps * (1 - eps) / M)
    assert abs(2 * norm.sf(norm.ppf(1 - eps / (2 * T))) * T - eps) < 1e-12
    count = int(joint_bad.sum())
    upper = float(beta.ppf(0.95, count + 1, M - count))
    CHECKS["시점별 95% 범위의 구간 전체 위반 빈도"] = float(point_path)
    CHECKS["위험 배분 범위의 구간 위반 빈도"] = float(joint_bad.mean())
    CHECKS["위험 배분 검증 빈도의 95% 상측 신뢰한계"] = upper
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    ax[0].plot(point, label="시점별 95%")
    ax[0].plot(joint, label="구간 전체 위험 배분")
    ax[0].set(xlabel="예측 단계", ylabel="양측 오차 여유")
    ax[0].legend()
    ax[1].bar(
        ["시점별 범위의\n구간 위반", "위험 배분의\n구간 위반"],
        [point_path, joint_bad.mean()],
    )
    ax[1].axhline(0.05, color="gray", ls=":")
    ax[1].set(ylabel="한 번 이상 위반한 경로 비율")
    finish_figure("gaussian_chance")


demo_gaussian()


### 3.7.2 Stabilizing Conditions — 확률적 안정화 조건

**교재 인쇄 pp.248**

지속 잡음이 있으면 모든 궤적의 원점 수렴 대신 기대값, 평균 제곱, 정상 분포 등 적절한 안정성 개념을 정해야 합니다. 교재는 무계 외란에서 유계 종단 집합의 강인 불변성을 요구하는 어려움을 설명합니다. 다음 실습은 제약 없는 안정한 확률 오차계의 분산 수렴을 보여 주며 완전한 확률적 MPC 안정성 증명은 아닙니다.

$$
\mathbb E[e_{k+1}^2]=\rho^2\mathbb E[e_k^2]+\sigma_w^2,\qquad\sigma_\infty^2=\frac{\sigma_w^2}{1-\rho^2}
$$

In [ ]:
def demo_mean_square():
    M, T = 30000, 24
    rho = 0.5
    sig = 0.1
    E, _ = error_paths(np.random.default_rng(53), M, T, rho, sig, gaussian=True)
    variance = np.zeros(T + 1)
    for k in range(T):
        variance[k + 1] = rho * rho * variance[k] + sig * sig
    sampled = np.var(E, axis=0, ddof=1)
    standard_error = np.sqrt(2 / (M - 1)) * variance
    assert np.all(np.abs(sampled[1:] - variance[1:]) < 6 * standard_error[1:])
    stationary = sig * sig / (1 - rho * rho)
    CHECKS["확률 오차의 정상 분산"] = stationary
    CHECKS["마지막 표본 분산 상대 오차"] = abs(float(sampled[-1] / stationary - 1))
    plt.figure()
    plt.plot(variance, label="해석적 분산")
    plt.plot(sampled, "o", ms=3, label="표본 분산")
    plt.axhline(stationary, color="gray", ls=":", label="정상 분산")
    plt.xlabel("시간 단계")
    plt.ylabel("오차 분산")
    plt.legend()
    finish_figure("mean_square")


demo_mean_square()


### 3.7.3 Stochastic Optimization — 확률적 최적화

**교재 인쇄 pp.248–249**

시나리오 설계는 표본 외란 경로에 대해 제약을 부과하여 계산 가능한 문제로 바꿉니다. ε는 새 경로의 위반 확률 목표이고 β는 표본으로 설계한 해가 그 목표를 만족하지 못할 확률입니다. 독립적인 검증 경로의 경험 빈도는 또 다른 표본 통계입니다. 아래에서는 구간 최대 절대 오차의 최댓값을 여유로 선택하는 스칼라 시나리오 문제를 풉니다.

$$
M\ge\frac{2}{\epsilon}\left(\log\frac1\beta+d\right),\qquad m=\max_{r=1,\ldots,M}\max_{j=0,\ldots,N}|e_j^{(r)}|
$$

In [ ]:
def demo_scenarios():
    margin, M = scenario_margin()
    T = 12
    test = 20000
    e, _ = error_paths(np.random.default_rng(1001), test, T)
    maxima = np.max(np.abs(e), axis=1)
    bad = maxima > margin
    k = int(bad.sum())
    upper = binomial_upper(k, test)
    assert 0 < margin < 0.2 and (1 - 0.05) ** M <= 0.01
    CHECKS["시나리오 설계 표본 수"] = M
    CHECKS["시나리오 오차 여유"] = margin
    CHECKS["독립 경로 위반 빈도"] = float(bad.mean())
    CHECKS["독립 검증의 95% 상측 신뢰한계"] = upper
    plt.figure()
    plt.hist(maxima, bins=50, density=True, alpha=0.7, label="독립 경로의 최대 오차")
    plt.axvline(margin, color="tab:orange", label="표본 설계 여유")
    plt.axvline(0.2, color="gray", ls=":", label="강인 반경")
    plt.xlabel("12단계 최대 절대 오차")
    plt.ylabel("확률 밀도")
    plt.legend()
    finish_figure("scenario_margin")
    print(
        "설계 표본 수:", M, "독립 검증 위반:", k, "/", test, "95% 상측 신뢰한계:", upper
    )


demo_scenarios()


**보충 실습 — 설계 실패 확률과 검증 신뢰한계**

한 경로의 최대 오차를 $Y$라 하고 독립 설계 경로 $M$개의 최대값을 여유로 택하면, 연속 분포에서 설계의 실제 위반 확률이 $\epsilon$보다 클 확률은 $(1-\epsilon)^M$입니다. 따라서 이 특별한 **스칼라 최대값 문제**의 최소 표본 수는 $\lceil\log\beta/\log(1-\epsilon)\rceil$입니다. 일반 시나리오 최적화에 그대로 적용하지 않습니다.

또한 독립 검증 $n$회에서 위반이 0회여도 95% 단측 신뢰상한은 $1-0.05^{1/n}$입니다. 위반 0회를 확률 0으로 보고하지 않도록 공통 함수에서 0회와 전부 위반한 경우까지 처리합니다. 설계에 사용한 데이터로 이 독립 검증을 대신하면 안 됩니다.

In [ ]:
def demo_sampling_confidence():
    eps = 0.05
    betas = np.array([0.1, 0.01, 0.001, 0.0001])
    exact = np.ceil(np.log(betas) / np.log1p(-eps)).astype(int)
    sufficient = np.ceil(2 / eps * (np.log(1 / betas) + 1)).astype(int)
    assert np.all((1 - eps) ** exact <= betas)
    assert np.all((1 - eps) ** (exact - 1) > betas)
    ns = np.array([20, 50, 100, 500, 1000])
    upper = np.array([binomial_upper(0, int(m)) for m in ns])
    assert np.allclose(upper, 1 - 0.05 ** (1 / ns)) and binomial_upper(10, 10) == 1
    CHECKS["최대값 설계 epsilon 0.05 beta 0.01 최소 표본"] = int(exact[1])
    CHECKS["검증 100회 무위반의 95% 상한"] = float(upper[2])
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    ax[0].plot(-np.log10(betas), sufficient, "o-", label="교재 충분조건 d=1")
    ax[0].plot(-np.log10(betas), exact, "o-", label="단일 최대값의 정확식")
    ax[0].set(xlabel="설계 신뢰 수준: -log10(beta)", ylabel="필요 설계 경로 수")
    ax[0].legend()
    ax[1].plot(ns, 100 * upper, "o-")
    ax[1].set(xlabel="독립 검증 경로 수 (모두 무위반)", ylabel="95% 위반 확률 상한 (%)")
    finish_figure("sampling_confidence")


demo_sampling_confidence()


**보충 실습 — 가장 작은 표본 여유만 고르면 설계 실패 확률이 커집니다**

앞의 $(1-\epsilon)^M$은 **한 번 설계한 최대값 여유**의 실패 확률입니다. 같은 분포에서 독립 표본 묶음 $L$개를 만들고 가장 작은 여유를 선택하면, 하나라도 과소 설계된 묶음이 있는 사건을 선택합니다. 따라서
$$p_1=(1-\epsilon)^M,\qquad p_{\rm select}=1-(1-p_1)^L.$$
$\epsilon=0.05$, $\beta=0.01$, $M=90$은 단일 설계에는 충분하지만, $L=20$개 중 최소 여유를 고르면 실패 확률이 약 18%가 됩니다. 이를 막는 한 방법은 각 후보에 $\beta/L$을 배분하여 $M\ge\lceil\log(\beta/L)/\log(1-\epsilon)\rceil$로 늘리는 것입니다. 합집합 상계는 후보 간 독립성 없이도 성립하지만 위 정확식은 독립 묶음에서만 성립합니다.

계산을 분명히 하기 위해 경로 점수 $Y\sim U[0,1]$인 별도 보충 문제를 풉니다. 여유 $m$의 실제 위반 확률은 정확히 $1-m$이고 최대값의 CDF는 $P(m\le t)=t^M$입니다. 따라서 $U^{1/M}$로 최대값을 직접 생성할 수 있습니다. 각 반복은 **새 설계 실험**이며, 한 설계에서 발생하는 외란 위반 빈도와 구분합니다. 연속 일반 분포도 확률 적분 변환으로 같은 순위 결과를 얻지만 이 셀은 MPC 궤적 자체를 시뮬레이션하지 않습니다.

실무에서는 후보 선택을 끝낸 뒤 손대지 않은 독립 데이터로 최종 설계를 검증합니다. 검증 결과를 보고 후보를 계속 바꾸면 그 검증 데이터도 선택에 사용된 것이므로, 고정 설계에 대한 신뢰한계를 그대로 재사용할 수 없습니다.


In [ ]:
def demo_scenario_selection():
    eps, beta_target, candidates, repeats = 0.05, 0.01, 20, 50000
    base_m = int(np.ceil(np.log(beta_target) / np.log1p(-eps)))
    safe_m = int(np.ceil(np.log(beta_target / candidates) / np.log1p(-eps)))
    rng = np.random.default_rng(3403)
    exact, empirical = [], []
    for samples, count in [(base_m, 1), (base_m, candidates), (safe_m, candidates)]:
        # Independent candidate maxima, using F_max(t)=t**samples.
        maxima = rng.random((repeats, count)) ** (1.0 / samples)
        chosen = maxima.min(axis=1)
        observed = float(np.mean(1.0 - chosen > eps))
        one_failure = (1.0 - eps) ** samples
        probability = float(-np.expm1(count * np.log1p(-one_failure)))
        se = np.sqrt(probability * (1.0 - probability) / repeats)
        assert abs(observed - probability) < 5 * se + 1 / repeats
        exact.append(probability)
        empirical.append(observed)
    assert exact[0] <= beta_target < exact[1]
    assert exact[2] <= candidates * (1 - eps) ** safe_m <= beta_target
    assert (1 - eps) ** (base_m - 1) > beta_target
    labels = [f"한 번 설계\nM={base_m}", f"{candidates}개 중 최소\nM={base_m}",
              f"위험 예산 보정\nM={safe_m}, L={candidates}"]
    pos = np.arange(3)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(pos - 0.18, exact, width=0.36, label="정확한 설계 실패 확률")
    ax.bar(pos + 0.18, empirical, width=0.36, label="독립 설계 반복 빈도")
    ax.axhline(beta_target, color="black", ls="--", label="설계 실패 목표 β=0.01")
    ax.set(xticks=pos, xticklabels=labels, ylabel="P(선택한 여유의 위반 확률 > ε)")
    ax.legend(fontsize=9)
    finish_figure("scenario_selection")
    CHECKS.update({
        "selection_single_samples": base_m,
        "selection_single_failure": exact[0],
        "selection_uncorrected_failure": exact[1],
        "selection_uncorrected_empirical": empirical[1],
        "selection_corrected_samples": safe_m,
        "selection_corrected_failure": exact[2],
    })
    print("단일 / 선택 / 보정 설계의 정확한 실패 확률:", np.array(exact))
    print("독립 설계 반복 빈도:", np.array(empirical))


demo_scenario_selection()


### 3.7.4 Tube-Based Stochastic MPC for Linear Constrained Systems — 선형 제약 시스템의 확률적 튜브 MPC

**교재 인쇄 pp.249–257**

교재의 이 구성은 유계 확률 외란을 사용합니다. 입력은 강인 오차 반경으로 축소하여 항상 한계를 지키고, 상태는 표본 설계 여유로 완화합니다. 명목 종단 상태를 강인하게 안전한 집합에 넣으면 이후 상태 제약을 강인하게 보장할 수 있습니다. 아래에서는 초기 시점에 길이 12의 명목 계획을 구해 이후 0으로 연장하며, 실제 상태로 명목 중심을 재설정하지 않습니다.

$$
|v_j|\le u_{\max}-|K|s,\qquad |z_j|\le x_{\max}-m,\qquad z_N=0
$$

In [ ]:
def demo_stochastic_tube():
    a, b, K, wb, rho, s, X, U, P, Kf = tube_parameters()
    N = 12
    T = 30
    margin, M = scenario_margin()
    x0 = 2.81
    assert X - s < x0 < X - margin
    z, v, _ = nominal_qp(x0, N=N, xlim=X - margin, terminal_zero=True)
    Z = np.r_[z, np.zeros(T - N)]
    V = np.r_[v, np.zeros(T - N)]
    E, W = error_paths(np.random.default_rng(203), 5000, T)
    actual = Z + E
    inputs = V + K * E[:, :-1]
    assert np.max(np.abs(inputs)) <= U + 2e-6
    assert np.max(np.abs(actual[:, N:])) <= X + 2e-6
    recurrence = a * actual[:, :-1] + b * inputs + W
    assert np.max(np.abs(actual[:, 1:] - recurrence)) < 2e-6
    violation = np.any(np.abs(actual[:, : N + 1]) > X + 2e-6, axis=1)
    envelope_bad = np.any(np.abs(E[:, : N + 1]) > margin, axis=1)
    assert np.all(~violation | envelope_bad)
    CHECKS["확률 튜브 실제 상태 위반 빈도"] = float(violation.mean())
    CHECKS["확률 튜브 실제 입력 최대 크기"] = float(np.max(np.abs(inputs)))
    CHECKS["강인 축소 대비 명목 상태 한계 확장"] = s - margin
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    tt = np.arange(T + 1)
    ax[0].plot(actual[:60].T, lw=0.5, alpha=0.25)
    ax[0].plot(Z, color="black", label="명목 중심")
    ax[0].axvline(N, color="gray", ls=":", label="명목 종단")
    ax[0].set(xlabel="시간 단계", ylabel="상태")
    ax[0].legend()
    ax[1].bar(["강인 상태 한계", "확률적 상태 한계"], [X - s, X - margin])
    ax[1].axhline(x0, color="tab:orange", ls=":", label="초기상태 2.81")
    ax[1].set(ylim=(2.75, 2.9), ylabel="명목 상태의 양의 상한")
    ax[1].legend()
    finish_figure("stochastic_tube")


demo_stochastic_tube()


**보충 실습 — 같은 설계를 다른 외란 분포에 쓰면**

기존 시나리오 설계는 독립 균등 외란 $[-0.1,0.1]$을 가정합니다. 같은 지지 집합이어도 확률이 양 끝점에 집중되면 경로 최대 오차의 분포가 달라집니다. 설계 여유를 고정하고 균등 외란, 양 끝점 외란, 범위가 두 배인 균등 외란에 각각 독립 검증합니다.

양 끝점 외란은 강인 반경 0.2 안에 있으므로 **강인 오차 집합**은 유효하지만, 균등 분포에서 설계한 **확률적 여유**의 보장은 옮겨지지 않습니다. 범위가 두 배이면 강인 반경 자체도 재설계해야 합니다. 아래 위반 사건은 실제 상태 제약이 아니라 12단계 **오차 여유의 초과**입니다.

In [ ]:
def demo_distribution_shift():
    margin, _ = scenario_margin()
    M = 20000
    T = 12
    rng = np.random.default_rng(932)
    cases = [
        rng.uniform(-0.1, 0.1, (M, T)),
        rng.choice([-0.1, 0.1], (M, T)),
        rng.uniform(-0.2, 0.2, (M, T)),
    ]
    rates = []
    high = []
    robust = []
    for w in cases:
        e = np.zeros(M)
        maximum = np.zeros(M)
        for k in range(T):
            e = 0.5 * e + w[:, k]
            maximum = np.maximum(maximum, np.abs(e))
        count = int(np.sum(maximum > margin))
        rates.append(count / M)
        high.append(binomial_upper(count, M))
        robust.append(float(np.mean(maximum > 0.2)))
    assert robust[0] == 0 and robust[1] == 0 and robust[2] > 0
    assert rates[1] > 0.05 and rates[2] > 0.05
    CHECKS["동일 범위 다른 분포의 오차 여유 초과 빈도"] = rates[1]
    CHECKS["두 배 외란의 강인 반경 초과 빈도"] = robust[2]
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    labels = ["균등 ±0.1", "양 끝점 ±0.1", "균등 ±0.2"]
    ax[0].bar(labels, rates, yerr=[np.zeros(3), np.array(high) - rates], capsize=4)
    ax[0].axhline(0.05, color="gray", ls=":")
    ax[0].set(ylabel="표본 설계 오차 여유 초과 비율")
    ax[1].bar(labels, robust)
    ax[1].set(ylabel="강인 반경 0.2 초과 비율")
    finish_figure("distribution_shift")


demo_distribution_shift()


## 3.8 Notes — 문헌 해설

**교재 인쇄 pp.257–262**

관련 연구를 읽을 때 불확실성 집합, 정책의 매개변수화, 강인·확률적 제약, 온라인·오프라인 계산을 비교합니다. 교재의 문헌 논의는 해당 판의 시점을 기준으로 하며 현재 기법 전체를 평가하는 자료는 아닙니다.

## 3.9 Exercises — 연습문제

**교재 인쇄 pp.262–268**

교재 연습문제 3.2는 최대값 차이 부등식, 3.3은 튜브 중심 제약의 동치 표현, 3.6은 강인 MPC 시뮬레이션을 다룹니다. 원본 문제는 교재의 번호로 확인하세요. 아래에는 이번 보충 예제를 확장하는 과제와 힌트를 제공합니다.

## 보충 과제와 힌트

1. 외란 반경을 0.1에서 0.2로 바꾸면 강인 오차 집합과 명목 입력 한계는 어떻게 바뀌나요?
2. 보조 이득을 $K=-0.3,-0.6,-0.9$로 바꿔 오차 반경과 입력 여유를 비교하세요. 오차 반경이 작은 이득이 항상 가장 유리한가요?
3. 최소–최대 실습에서 제어자가 현재 외란을 먼저 알고 입력을 고르면 어떤 다른 문제를 풀게 되나요?
4. 최적화한 중심을 사용하지 않고 매번 $z=x$로 재설정하면 어떤 실행 가능성 조건을 잃을 수 있나요?
5. 각 시점의 위반 확률 5%와 20단계 전체의 위반 확률 5%를 구분하고 합집합 상계를 계산하세요.
6. 시나리오 설계에서 $\epsilon$을 고정하고 $\beta$를 줄이면 표본 수와 설계의 의미는 어떻게 달라지나요?

<details><summary>힌트</summary>

1. $s=w_{\max}/(1-|a+bK|)$와 $u_{\max}-|K|s$를 다시 계산하세요. 종단 입력·불변 조건도 다시 확인해야 합니다.
2. 상태 여유뿐 아니라 $|K|s$의 보정 입력 여유와 명목 비용을 함께 봅니다.
3. $\min_u\max_w$와 $\max_w\min_u$의 정보 순서는 다릅니다.
4. 실제 상태가 원래 제약 안에 있어도 축소된 명목 실행 가능 집합 바깥일 수 있습니다.
5. 개별 5%만 알면 합집합 상계는 $\min(1,20\times0.05)=1$로 유용하지 않습니다. 동일 배분은 시점별 0.25%입니다.
6. $\beta$는 외란 경로의 위반 확률이 아니라 표본으로 만든 설계가 목표를 놓칠 확률입니다. 표본 수는 대수 항을 통해 증가합니다.

</details>

## 실행 검증 결과

기본 모델·외란 가정과 수치 허용오차 아래 검사한 지표입니다. 표본 결과를 모든 외란에 대한 증명으로 바꾸어 해석하지 않습니다.

In [ ]:
for name, value in CHECKS.items():
    print(f"{name}: {value:.8g}")
print("검증 지표 수:", len(CHECKS))
print(
    {
        "Python": sys.version.split()[0],
        "NumPy": np.__version__,
        "SciPy": scipy.__version__,
        "Matplotlib": matplotlib.__version__,
        "CVXPY": cp.__version__,
    }
)


## 참고 자료

- Rawlings, Mayne, Diehl (2017), *Model Predictive Control: Theory, Computation, and Design*, 2nd edition, 1st printing, Chapter 3, pp.193–268.
- 표본 수의 충분조건은 교재 §3.7.3의 식 (3.23)을 사용했습니다. 단일 최대값 설계에는 별도로 순서통계량의 정확한 실패 확률 경계를 확인합니다.
- §3.1.2와 Example 3.13은 원본 재현이고 그 외는 보충 실습입니다. 가우스 예제와 유계 균등 외란 예제를 구분합니다.
- 95% 상측 신뢰한계는 고정된 설계를 독립 검증한 이항 비율의 단측 Clopper–Pearson 경계입니다. 설계의 β와는 다른 확률입니다.
- 한글 글꼴은 Google Fonts의 나눔고딕, 수학·영문 글꼴은 DejaVu Sans를 사용합니다.